Adding the libraries

In [34]:
import pandas as pd

Basic statistics of dataset

In [35]:
Full_data = pd.read_csv("iran_poi_raw.csv")
Total_records = Full_data.shape[0]
print()
print(f'Total records of dataset: {Total_records}')

Categories = [
    "amenity" , "shop" , "leisure" , "office" ,
    "tourism" , "craft" , "healthcare"
]
Counts = Full_data["main_key"].value_counts()
for category in Categories:
    count = Counts.get(category , 0)
    precentage = count / Total_records
    print(f"{category} => count: {count} | precentage : {precentage:.2f}")

missing_count = Full_data.isna().sum()
missing_percentage = (missing_count / Total_records) * 100

missing_report = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage (%)": missing_percentage.round(2)
})

print("\nMissing Values Report:")
print(missing_report.to_string())


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18760\3994022593.py:1: DtypeWarning: Columns (0: capacity) have mixed types. Specify dtype option on import or set low_memory=False.
  Full_data = pd.read_csv("iran_poi_raw.csv")



Total records of dataset: 297152
amenity => count: 138493 | precentage : 0.47
shop => count: 72122 | precentage : 0.24
leisure => count: 42320 | precentage : 0.14
office => count: 25972 | precentage : 0.09
tourism => count: 12220 | precentage : 0.04
craft => count: 3911 | precentage : 0.01
healthcare => count: 2114 | precentage : 0.01

Missing Values Report:
                  Missing Values  Missing Percentage (%)
osm_type                       0                    0.00
osm_id                         0                    0.00
version                        0                    0.00
edit_date                      0                    0.00
lat                            0                    0.00
lon                            0                    0.00
main_key                       0                    0.00
main_value                     0                    0.00
name                       73574                   24.76
name:fa                   289881                   97.55
name:en    

City names

In [36]:
cities = Full_data["addr:city"]

unique_cities = cities.nunique(dropna=True)
missing_cities = cities.isna().sum()

print("Unique city values:", unique_cities)
print("Missing city values:", missing_cities)



print("\nMost frequent city values:")
print(cities.value_counts().head(20).to_string())


tehran_variants = [
    "تهران",
    "Tehran",
    "شهر تهران",
    "tehran",
    "TEHRAN"
]

print("\nWritten forms of Tehran:")

for city in tehran_variants:
    count = (cities == city).sum()
    print(f"{city}: {count}")

Unique city values: 2525
Missing city values: 252005

Most frequent city values:
addr:city
تهران        6110
شیراز        4939
اصفهان       1194
مشهد         1125
Tehran       1018
شهر تهران     932
بابل          932
کرج           835
تبریز         821
چهارفریضه     780
فولادشهر      600
کیش           521
اقلید         481
اهواز         479
اردبیل        473
بروجرد        452
tehran        408
قائمشهر       395
بندرعباس      387
رشت           387

Written forms of Tehran:
تهران: 6110
Tehran: 1018
شهر تهران: 932
tehran: 408
TEHRAN: 8


Arabic names

In [37]:
arabic_chars = Full_data["name"].str.contains("[يك]", na=False)

total_names = arabic_chars.sum()
unique_names = Full_data.loc[arabic_chars, "name"].nunique()

print("Total records with Arabic characters:", total_names)
print("Unique names with Arabic characters:", unique_names)


Total records with Arabic characters: 1726
Unique names with Arabic characters: 1697


Capacity value

In [45]:
capacity = Full_data["capacity"].astype("string").str.strip()

missing_value = capacity.isna()

numeric_capacity = pd.to_numeric(capacity, errors="coerce")

invalid = ~missing_value & numeric_capacity.isna()

negative = numeric_capacity < 0


print("Missing capacity values:", missing_value.sum())
print("Valid non-negative numeric values:", (numeric_capacity.notna() & ~negative).sum())
print("Invalid non-numeric values:", invalid.sum())
print("Negative values:", negative.sum())


print("\nNegative capacity values:")
print(Full_data.loc[negative, ["capacity"]].value_counts().to_string())


print("\nInvalid non-numeric values:")
print(Full_data.loc[invalid, ["capacity"]].value_counts().head(10).to_string())

Missing capacity values: 296298
Valid non-negative numeric values: 842
Invalid non-numeric values: 10
Negative values: 2

Negative capacity values:
capacity
-1.0        1
-50.0       1

Invalid non-numeric values:
capacity             
۵ نفر                    1
۵۰                       1
۲۰                       1
۶۰                       1
۱۰                       1
۸۰۰متر مربع              1
مخزن (تانکر) آب حسینی    1
مشهد                     1
ضض300                    1
ترمینال                  1


Phone number

In [ ]:
phones = Full_data["phone"].astype("string").str.strip()

phones = phones[phones != ""]

persian_digits = "۰۱۲۳۴۵۶۷۸۹"


def get_phone_format(phone):
    has_persian = any(ch in persian_digits for ch in phone)
    has_latin = any(ch in "0123456789" for ch in phone)

    if has_persian and has_latin:
        digit_type = "Mixed Persian and Latin"
    elif has_persian:
        digit_type = "Persian digits"
    else:
        digit_type = "Latin digits"

    if phone.startswith("+98") or phone.startswith("+۹۸") :
        country_code = "With +98"
    elif phone.startswith("0098") or phone.startswith("۰۰۹۸"):
        country_code = "With 0098"
    elif phone.startswith("98") or phone.startswith("۹۸"):
        country_code = "With 98"
    else:
        country_code = "Without country code"

    spacing = "With spaces" if " " in phone else "Without spaces"

    digits = "".join(
        ch for ch in phone
        if ch in persian_digits + "0123456789"
    )

    if len(digits) < 10:
        length_type = "Possibly incomplete"
    elif len(digits) == 10:
        length_type = "10 digits"
    else:
        length_type = "+10 digits"
            

    return digit_type, country_code, spacing, length_type


phone_formats = phones.apply(get_phone_format)

phone_table = pd.DataFrame({
    "phone": phones,
    "digit_type": [x[0] for x in phone_formats],
    "country_code": [x[1] for x in phone_formats],
    "spacing": [x[2] for x in phone_formats],
    "length_type": [x[3] for x in phone_formats]
})


examples = phone_table.drop_duplicates(
    subset=["digit_type", "country_code", "spacing", "length_type"]
).head(20)

print(examples.to_string(index=False))


TypeError: 'bool' object is not iterable